In [1]:
!pip install torch torchvision

In [2]:
import torch
import torchvision
import torch.nn.functional as F  # Parameterless functions, like (some) activation functions
import torchvision.datasets as datasets  # Standard datasets
import torchvision.transforms as transforms  # Transformations we can perform on our dataset for augmentation
from torch import optim  # For optimizers like SGD, Adam, etc.
import torch.nn as nn  # All neural network modules
from torch.utils.data import DataLoader
from tqdm import tqdm  # For nice progress bar!
import matplotlib.pyplot as plt

Setting the device
-

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

Pretrained Model
-

VGG16 has already learned to identify edges, textures, and shapes from millions of images in ImageNet

In [4]:
num_classes = 2
model = torchvision.models.vgg16(weights="DEFAULT")
model.avgpool = nn.Identity()
# model.classifier = nn.Sequential(
#     nn.Linear(512,100), nn.ReLU(),nn.Linear(100,50),nn.ReLU(),nn.Linear(50,num_classes) # 32*32
# )
# Updated Classifier for 75x75 input
model.classifier = nn.Sequential(
    nn.Linear(512 * 2 * 2, 100),  # Changed from 512 to 2048 (512*4)
    nn.ReLU(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, num_classes)
)
model.to(device)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

Hyper-Parameters
-

In [5]:
# Updated Hyper-parameters
in_channels =3
learning_rate = 3e-4 #karpathy's constant
num_epochs = 3
batch_size =4

Load the Data
-

Input Size, Conv Base Output Calculation,   Final Feature Map (Channels * H * W)
1. 32,       |      32/32=1,             |           512×1×1=512
2. 75,       |    75/32=2.34→2,          |           512×2×2=2048

In [6]:
trans_train = transforms.Compose([transforms.Resize(75), # 32 * 32
                                  transforms.RandomRotation(30),
                                  transforms.RandomResizedCrop(75),
                                  transforms.RandomHorizontalFlip(),
                                  transforms. ToTensor()])

trans_valid = transforms.Compose([transforms.Resize(75),
                                  transforms.CenterCrop(75),
                                   transforms.ToTensor()])

data_dir = "C:/Users/dbda.STUDENTSDC/Desktop/Practical Machine Learning/Datasets/Cats and Dogs"
train_dataset = datasets.ImageFolder(data_dir + '/train', transform=trans_train)
test_dataset = datasets.ImageFolder(data_dir + '/val', transform=trans_valid)

batch_size = 1
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle= True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle =False)
type(train_dataset)

torchvision.datasets.folder.ImageFolder

Loss and optimizer
-

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Train Network
--

In [8]:
for epoch in range(num_epochs):
    loop =tqdm(train_loader,leave=True)
    for batch_idx, (data, targets) in enumerate(loop):
        # Get data to cuda if possible
        data = data.to(device=device)
        targets = targets.to(device=device)
        targets = F.one_hot(targets,num_classes=num_classes).float()
        # forward
        scores = model(data)
        loss = criterion(scores, targets)
        
        # backward
        optimizer.zero_grad()
        loss.backward()

        # gradient descent or adam step
        optimizer.step()

        loop.set_description(f"Epoch: [{epoch+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())

Epoch: [3/3]: 100%|██████████| 275/275 [00:54<00:00,  5.06it/s, loss=0.476]


##### Check accuracy on training & test to see how good our model

In [9]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)

            scores = model(x)
            _, predictions = scores.max(1)
            # predictions = (scores[:,1]>0.5).long()
            num_correct += (predictions == y).sum()
            num_samples += predictions.size(0)
        
    model.train()
    return float(num_correct) / num_samples


In [10]:
# Check accuracy on both sets
train_acc = check_accuracy(train_loader, model)
test_acc = check_accuracy(test_loader, model)

print(f"Accuracy on training set: {train_acc*100:.2f}%")
print(f"Accuracy on test set: {test_acc*100:.2f}%")

Accuracy on training set: 65.45%
Accuracy on test set: 65.71%


For 32*32 
1. Accuracy on training set: 65.45%
2. Accuracy on test set: 65.71%